In [46]:
import pandas as pd
import numpy as np
import random
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import io
import logging
from datetime import datetime, timedelta

logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s')



### 1. Carregar o Dataset

In [66]:
url = "https://www.bcb.gov.br/pda/desig/desenrola/dados_desenrola.csv"
local_file = "../data/raw/dados_desenrola.csv"
csv_config_br = {"sep": ";", "decimal": ",", "encoding": "utf-8"}

def extract_data(url: str, local_file: str, csv_config: dict) -> pd.DataFrame:
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    try:
        logging.info("Tentando baixar o dataset online...")
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()
        logging.info("Download bem-sucedido. Carregando os dados...")
        df = pd.read_csv(
            io.StringIO(response.text),
            **csv_config
            )
        logging.info(f"Dados extraídos com sucesso da internet. [{len(df)}] registros encontrados.")
    except Exception as e:
        logging.error(f"Não foi possível baixar da internet. Motivo: {e}")
        logging.info("Utilizando arquivo local como alternativa...")
        df = pd.read_csv(
            local_file,
            **csv_config
            )
        logging.info(f"Dados extraídos com sucesso de arquivo local. [{len(df)}] registros encontrados.")
    return df

df_raw = extract_data(url, local_file, csv_config_br)
df_raw.info()


INFO - Tentando baixar o dataset online...
INFO - Download bem-sucedido. Carregando os dados...
INFO - Dados extraídos com sucesso da internet. [10598] registros encontrados.


<class 'pandas.DataFrame'>
RangeIndex: 10598 entries, 0 to 10597
Data columns (total 7 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   DATA_BASE                     10598 non-null  int64  
 1   TIPO_DESENROLA                10598 non-null  int64  
 2   UNIDADE_FEDERACAO             10598 non-null  str    
 3   COD_CONGLOMERADO_FINANCEIRO   10598 non-null  int64  
 4   NOME_CONGLOMERADO_FINANCEIRO  10598 non-null  str    
 5   NUMERO_OPERACOES              10598 non-null  int64  
 6   VOLUME_OPERACOES              10598 non-null  float64
dtypes: float64(1), int64(4), str(2)
memory usage: 579.7 KB


#### 1.5 Sujar o Dataset para limpeza posterior

In [82]:
def dirty_data(df, noise_level=0.02):
    # Cria uma cópia "suja" de parte do dataframe original (preservando os dados limpos) e adiciona essa cópia ao final do dataframe original.
    df_clean = df.copy()
    corrupted_rows = []
    
    # Introduz valores nulos
    df_nulls = df_clean.sample(frac=noise_level).copy()
    for col in df_nulls.columns:
        df_nulls.loc[df_nulls.sample(frac=0.5).index, col] = np.nan
    corrupted_rows.append(df_nulls)
    
    # Introduz espaços em branco em colunas de texto
    df_text_columns = df_clean.select_dtypes(include=['object', 'string']).columns
    if not df_text_columns.empty:
        df_spaces = df_clean.sample(frac=noise_level).copy()
        for col in df_text_columns:
            df_spaces[col] = '   ' + df_spaces[col].astype(str) + '   '
        corrupted_rows.append(df_spaces)
        
    # Introduz datas inválidas
    df_dates = df_clean.sample(frac=noise_level).copy()
    df_dates['DATA_BASE'] = 'invalid_date'
    corrupted_rows.append(df_dates)
        
    # Introduz outliers em colunas numéricas
    numeric_columns = df_clean.select_dtypes(include=np.number).columns
    if not numeric_columns.empty:
        df_outliers = df_clean.sample(frac=noise_level).copy()
        for column in numeric_columns:
            df_outliers[column] = df_clean[column].max() * 10
        corrupted_rows.append(df_outliers)
        
    # Introduz duplicatas
    df_duplicates = df_clean.sample(frac=noise_level).copy()
    corrupted_rows.append(df_duplicates)
    
    df_dirty = pd.concat([df_clean] + corrupted_rows, ignore_index=True)

    # Converte as colunas de inteiros para Int64 para permitir valores nulos, mantendo a consistência de tipo com o dataframe original.
    int_cols = df_clean.select_dtypes(include='int64').columns
    int_cols = [col for col in int_cols if col != 'DATA_BASE'] # Filtra e remove a coluna de data
    for col in int_cols:
        df_dirty[col] = df_dirty[col].astype('Int64')

    return df_dirty

df_dirty = dirty_data(df_raw)
os.makedirs("../data/raw", exist_ok=True)
df_dirty.to_csv("../data/raw/desenrola_sujo.csv", index=False)
print(f"Dataset gerado com {len(df_dirty)} registros.")
print(df_dirty.tail())


Dataset gerado com 11658 registros.
      DATA_BASE  TIPO_DESENROLA UNIDADE_FEDERACAO  \
11653    202411               1                GO   
11654    202603               1                MG   
11655    202312               1                SC   
11656    202310               1                PB   
11657    202409               1                PE   

       COD_CONGLOMERADO_FINANCEIRO NOME_CONGLOMERADO_FINANCEIRO  \
11653                        30379                    SANTANDER   
11654                        84693   NU PAGAMENTOS - PRUDENCIAL   
11655                        49944                  BTG PACTUAL   
11656                        51626      CAIXA ECONÔMICA FEDERAL   
11657                        51626      CAIXA ECONÔMICA FEDERAL   

       NUMERO_OPERACOES  VOLUME_OPERACOES  
11653                23           7467.62  
11654                 5           3233.61  
11655              1372        2110113.98  
11656               254         163477.30  
11657                 

### 2. Inspecionar o Dataset

In [81]:
df_dirty.info()

<class 'pandas.DataFrame'>
RangeIndex: 11658 entries, 0 to 11657
Data columns (total 7 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   DATA_BASE                     11552 non-null  object 
 1   TIPO_DESENROLA                11552 non-null  Int64  
 2   UNIDADE_FEDERACAO             11552 non-null  str    
 3   COD_CONGLOMERADO_FINANCEIRO   11552 non-null  Int64  
 4   NOME_CONGLOMERADO_FINANCEIRO  11552 non-null  str    
 5   NUMERO_OPERACOES              11552 non-null  Int64  
 6   VOLUME_OPERACOES              11552 non-null  float64
dtypes: Int64(3), float64(1), object(1), str(2)
memory usage: 671.8+ KB
